# Tree Traversals: BFS vs DFS

---

## The Big WHY

In binary_trees notebook we did recursive DFS traversals. Today we add two things:

1. **BFS (Breadth-First Search)** — visits nodes level by level using a queue
2. **Iterative DFS** — same order as recursive, but uses an explicit stack instead of the call stack

Why does this matter?

| | BFS | DFS |
|--|-----|-----|
| **Uses** | Queue | Stack (or recursion) |
| **Order** | Level by level | Branch by branch |
| **Best for** | Shortest path, level-based problems | Subtree problems, path problems |
| **Space** | O(w) — w = max width of tree | O(h) — h = height of tree |

For a wide, shallow tree → DFS uses less memory.  
For a deep, narrow tree → BFS uses less memory.  
For a balanced tree → BFS space is O(n/2) at the last level, DFS is O(log n).

**In AI/ML:** BFS is how graph neural networks propagate features layer by layer. DFS is how decision trees traverse from root to leaf during prediction.

---

## BFS — the core idea

Use a queue. Start with root. For each node you pop, add its children to the queue.

```
        3
       / \
      9  20
         / \
        15   7

Queue: [3]
Pop 3  → add 9, 20   → Queue: [9, 20]
Pop 9  → no children → Queue: [20]
Pop 20 → add 15, 7   → Queue: [15, 7]
Pop 15 → no children → Queue: [7]
Pop 7  → no children → Queue: []

Level order: [3, 9, 20, 15, 7]
```

---

## Iterative DFS and recuresive DFS

When we write recursive DFS, Python manages a call stack behind the scenes — every function call gets added to it, and when it returns it gets removed.

```
maxDepth(1)        ← pushed onto call stack
  maxDepth(2)      ← pushed
    maxDepth(4)    ← pushed, returns 0, popped
    maxDepth(5)    ← pushed, returns 0, popped
  returns 1, popped
  maxDepth(3)      ← pushed, returns 0, popped
returns 2, popped
```

we never see this stack — Python handles it automatically.

Iterative DFS does the exact same thing, but we create the stack ourselves using a plain list and manage it manually:

```python
stack = [root]
while stack:
    node = stack.pop()   # we control what gets processed next
    ...
```

```
Stack: [3]
Pop 3  → visit, push right(20) then left(9)  → Stack: [20, 9]
Pop 9  → visit, no children                  → Stack: [20]
Pop 20 → visit, push right(7) then left(15)  → Stack: [7, 15]
Pop 15 → visit, no children                  → Stack: [7]
Pop 7  → visit, no children                  → Stack: []

Preorder: [3, 9, 20, 15, 7]
```

Note: push **right first**, then left — because stacks are LIFO, left gets popped first.

---

In [2]:
# Paste TreeNode, build_tree, tree_to_list from binary_trees notebook here
from collections import deque

class TreeNode:
    def __init__(self, val, left = None, right = None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"
    
def build_tree(values):
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1

    while queue and i < len(values):
        node = queue.popleft()

        # left child
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1    # always increment i, even if we skip to match the index

        # right child
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            
            i += 1
    return root

def tree_to_list(root):
    queue = deque([root])
    result = []

    while queue:
        node = queue.popleft()
        if node is not None:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)

    while result and result[-1] is None:
        result.pop()

    return result

## Part 1 — BFS / Level Order Traversal

Return a list of lists — each inner list is one level of the tree.

```
        3
       / \
      9  20
         / \
        15   7

Output: [[3], [9, 20], [15, 7]]
```

**Hint:** At each step, you need to know how many nodes are on the current level. `len(queue)` at the start of each level tells you exactly that — process that many nodes, then move to the next level.

In [7]:
# levelOrder(root) → list of lists
# Use a queue, process level by level
# len(queue) at the start of each level = number of nodes on that level
def levelOrder(root):
    if root is None:  # base case
        return [] 
    queue = deque([root])
    result = []

    while queue:
        level_size = len(queue) # gives how many nodes are on this level
        level = []

        for i in range(level_size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result

# Tests:
print(levelOrder(build_tree([3, 9, 20, None, None, 15, 7])))  #  [[3], [9, 20], [15, 7]]
print(levelOrder(build_tree([1])))                             #  [[1]]
print(levelOrder(None))                                        #  []

[[3], [9, 20], [15, 7]]
[[1]]
[]


## Part 2 — Iterative DFS (Preorder)

Same result as recursive preorder — Node, Left, Right — but using an explicit stack.

**Key rule:** Push right child first, then left child. Why? Stack is LIFO — you want left popped first.

In [9]:
# iterative_preorder(root) → list
# Use a stack (plain list), push right then left
def iterative_preorder(root):
    if root is None:
        return []
    
    stack = [root]
    result = []

    while stack:
        node = stack.pop()
        result.append(node.val)
        if node.right is not None:
            stack.append(node.right)
        if node.left is not None:
            stack.append(node.left)
    return result

# Test — should match recursive preorder:
iterative_preorder(build_tree([1, 2, 3, 4, 5]))   # [1, 2, 4, 5, 3]

[1, 2, 4, 5, 3]

---

## LeetCode

### Problem 1: Binary Tree Level Order Traversal (LC #102)

**What it asks:** Return node values grouped by level — same as Part 1 above.

We already built this. Just clean it up and make sure edge cases work.

In [10]:
# levelOrder(root) → List[List[int]]
# Already done in Part 1 — verify edge cases below
def levelOrder(root):
    # base case
    if root is None:
        return []
    
    queue = deque([root])
    result = []

    while queue:
        level = []
        level_size = len(queue)  # number of nodes - needed to loop through
        
        for i in range(level_size):  # to loop through a specific level and extract the nodes of that level
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)

    return result
        

# Tests:
print(levelOrder(build_tree([3, 9, 20, None, None, 15, 7])))  # [[3], [9, 20], [15, 7]]
print(levelOrder(build_tree([1])))                             # [[1]]
print(levelOrder(None))                                        # []

[[3], [9, 20], [15, 7]]
[[1]]
[]


### Problem 2: Same Tree (LC #100)

**What it asks:** Given two trees `p` and `q`, return `True` if they are identical — same structure and same values at every node.

**Think first:** Two trees are the same if at every node, three things are true simultaneously:
- Both are `None` → True
- One is `None`, the other isn't → False
- Both have the same value AND their left subtrees are the same AND their right subtrees are the same → True

That's the entire problem. Which traversal order is this?

In [11]:
# isSameTree(p, q) → bool
# Handle: both None, one None, values differ, recurse left and right
# it is a preorder traversal - we check the current node's value first then recurse into left, then right.

def isSameTree(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None:
        return False
    if p.val != q.val:
        return False
    left_check = isSameTree(p.left, q.left)
    right_check = isSameTree(p.right, q.right)
    return left_check and right_check

    # or use this one line instead of last 5 lines of code
    # return p.val == q.val and isSameTree(p.left, q.left) adn isSameTree(p.right, q.right)

# Tests:
p1 = build_tree([1, 2, 3])
q1 = build_tree([1, 2, 3])
print(isSameTree(p1, q1))   # True

p2 = build_tree([1, 2])
q2 = build_tree([1, None, 2])
print(isSameTree(p2, q2))   # False

p3 = build_tree([1, 2, 1])
q3 = build_tree([1, 1, 2])
print(isSameTree(p3, q3))   # False


True
False
False


---

## BFS vs DFS — When to Use Each

| Situation | Use |
|-----------|-----|
| Need level-by-level output | BFS |
| Shortest path in unweighted graph | BFS |
| Need to explore full subtrees | DFS |
| Path problems (root to leaf) | DFS |
| Tree is very wide | DFS (less memory) |
| Tree is very deep | BFS (less memory) |

---

## Summary

| Concept | Key idea |
|---------|----------|
| BFS | Queue, level by level, use `len(queue)` to track level boundaries |
| Iterative DFS | Stack, push right first then left |
| Level order | BFS — collect each level into its own list |
| Same Tree | Recurse both trees together, check value + structure |